# Question 1 — Cell Segmentation & Counting
Using `scikit-image` and watershed segmentation to detect and count cells in a blood smear image.

In [ ]:
# Step 0: Install dependencies (run once)
import subprocess
subprocess.run(["pip", "install", "scikit-image", "scipy", "matplotlib", "numpy"], check=True)

In [ ]:
# Step 1: Imports
import numpy as np
import matplotlib.pyplot as plt
from skimage import io, color, morphology, segmentation, measure, feature
from skimage.morphology import disk, remove_small_objects, remove_small_holes
from skimage.segmentation import watershed
from skimage.filters import threshold_otsu, gaussian
from scipy import ndimage as ndi

In [ ]:
# Step 2: Read image
img = io.imread("cells.jpg")   # change path if needed
gray = color.rgb2gray(img)
print(f"Image shape: {img.shape}")

In [ ]:
# Step 3: Preprocessing — smooth + Otsu threshold
# Cells are darker than the white background
blurred = gaussian(gray, sigma=2)
thresh  = threshold_otsu(blurred)
binary  = blurred < thresh
print(f"Otsu threshold: {thresh:.4f}")

In [ ]:
# Step 4: Morphological cleanup
binary = remove_small_objects(binary, min_size=200)      # remove debris
binary = remove_small_holes(binary, area_threshold=500)  # fill holes inside cells
binary = morphology.binary_closing(binary, disk(3))      # seal cell edges

In [ ]:
# Step 5: Watershed segmentation to separate touching cells
distance   = ndi.distance_transform_edt(binary)
coords     = feature.peak_local_max(distance, min_distance=18, labels=binary)
mask_peaks = np.zeros(distance.shape, dtype=bool)
mask_peaks[tuple(coords.T)] = True
markers, _ = ndi.label(mask_peaks)
labels     = watershed(-distance, markers, mask=binary)

In [ ]:
# Step 6: Count valid cells (filter tiny fragments < 300 px)
props       = measure.regionprops(labels)
valid_cells = [p for p in props if p.area >= 300]
cell_count  = len(valid_cells)

print("=" * 40)
print(f"  TOTAL CELLS DETECTED: {cell_count}")
print("=" * 40)

In [ ]:
# Step 7: Build binary output mask (1 = cell, 0 = background)
valid_labels  = set(p.label for p in valid_cells)
binary_output = np.isin(labels, list(valid_labels)).astype(np.uint8)

# Save binary image for submission
binary_img = (binary_output * 255).astype(np.uint8)
io.imsave("cells_binary.png", binary_img)
print("Saved: cells_binary.png  (255=cell, 0=background)")

In [ ]:
# Step 8: Visualize results
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(img)
axes[0].set_title("Original", fontsize=14, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(binary_output, cmap="gray")
axes[1].set_title("Binary Mask (1=cell, 0=background)", fontsize=14, fontweight="bold")
axes[1].axis("off")

overlay = segmentation.mark_boundaries(img, labels, color=(0, 1, 0), mode="thick")
axes[2].imshow(overlay)
for p in valid_cells:
    cy, cx = p.centroid
    axes[2].plot(cx, cy, "r.", markersize=3)
axes[2].set_title(f"Detected Cells: {cell_count}", fontsize=14, fontweight="bold", color="darkgreen")
axes[2].axis("off")

plt.tight_layout()
plt.savefig("cells_segmentation.png", dpi=150, bbox_inches="tight")
plt.show()
print("Visualization saved: cells_segmentation.png")